In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ \mathbb{R} \to \mathbb{R}, \quad y(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} $$

$$ \mathbb{R^n} \to \mathbb{R^n}, \quad y(z) = (y(z_1), y(z_2), \dots, y(z_n)) $$

$$ \text{Derivative} $$

$$ \frac{dy}{dz_i} = 1 - y_i^2 $$

$$ \frac{dy}{dz} = 1 - y \odot y $$

$$ \text{Jacobian} $$

$$
J_y(z) =
\begin{bmatrix}
    \frac{dy}{dz_1} & 0 & \cdots & 0 \\
    0 & \frac{dy}{dz_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{dy}{dz_n}
\end{bmatrix}
$$

$$ dy = J_y(z) \, dz $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dz}, dz \right \rangle =
\left \langle \frac{dF}{dy}, dy \right \rangle

\\[2em]

\left \langle \frac{dF}{dy}, dy \right \rangle = \\[0.5em]
\left \langle \frac{dF}{dy}, J_y(z) \, dz \right \rangle = \\[0.5em]
\left \langle {J_y(z)}^\top \, \frac{dF}{dy}, dz \right \rangle \implies \\[1em]

\frac{dF}{dz} = {J_y(z)}^\top \, \frac{dF}{dy} \\[0.5em]
\frac{dF}{dz} = \frac{dy}{dz} \odot \frac{dF}{dy}
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
from approx import approx # type: ignore


def tanh(z: tr.Tensor) -> tr.Tensor:
    """Hyperbolic tangent function."""

    return 2 * sigmoid.sigmoid(2 * z) - 1


class TanhFunction(autograd.Function):
    """Custom autograd function with `forward` and `backward` passes for the `tanh`."""

    @staticmethod
    def forward(ctx, z: tr.Tensor) -> tr.Tensor:
        y = tanh(z)
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, dF_dy: tr.Tensor) -> tuple[tr.Tensor, ]:
        (y, ) = ctx.saved_tensors

        dy_dz = 1 - y * y
        dF_dz = dy_dz * dF_dy

        return (dF_dz, )


class Tanh(nn.Module):
    """Custom module for the `tanh`."""

    def forward(self, z: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(z)


In [ ]:
# Unit tests

def test_basic() -> None:
    assert tanh(1000) == approx(tr.tensor(1.0))
    assert tanh(-1000) == approx(tr.tensor(-1.0))
    assert tanh(0) == approx(tr.tensor(0.0))
    assert tanh(1) == approx(tr.tensor(0.76159))
    assert tanh(-1) == approx(tr.tensor(-0.76159))


def test_gradcheck(samples) -> None:
    def func(z: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(z)

    z = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (z, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    z = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    z1 = z.clone().detach().requires_grad_(True)
    y1 = Tanh()(z1)
    y1.sum().backward()

    z2 = z.clone().detach().requires_grad_(True)
    y2 = tr.tanh(z2)
    y2.sum().backward()

    assert y1 == approx(y2)
    assert z1.grad == approx(z2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)